# 3. Fraud Message Classifier Model Training
This notebook outlines how to build a Machine Learning model using TF-IDF and Multinomial Naive Bayes to classify SMS / text messages as safe or fraudulent.

### Objective:
- Build a text classification pipeline for messages.
- Train the model on common Indian banking scam texts, lottery traps, and job offer frauds.
- Export the trained pipeline to `message_classifier.pkl` in the python server directory.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print("Imports complete.")


## 2. Dataset Generation
We construct a representative training corpus containing:
- **Ham Messages (Class 0)**: Normal personal notifications, greetings, or conversational texts.
- **Spam/Scam Messages (Class 1)**: Common text messages representing phishing, KYC account suspension threats, lottery winners, and online job task scams.

In [ ]:
data = {
    'text': [
        "Hey! Are we still meeting for lunch today?",
        "Your package has been delivered to your mailbox. Have a good day!",
        "Don't forget to submit the report by 5 PM today.",
        "Congratulations on completing your graduation! So proud of you.",
        "Hello, just wanted to check if you got my email about the project.",
        "Let's catch up this weekend. Let me know when you're free.",
        "The weather is beautiful today, we should go for a walk.",
        "Can you send me the password for the office WiFi?",
        "Please bring some milk on your way back home.",
        "Thanks for the help yesterday, I really appreciate it.",
        "URGENT: Your SBI bank account is blocked due to missing KYC. Click here to verify: http://sbi-kyc-check.net",
        "Congratulations! You won a lottery of Rs 1 Crore in KBC. Call +91-9988776655 to claim your prize money.",
        "Electricity connection will be disconnected tonight at 9:30 PM due to past unpaid bills. Contact officer at 8822991100.",
        "Earning opportunity: Earn Rs 5000/day working part-time from home. Just like YouTube videos. Join telegram: http://t.me/earnparttime",
        "Dear customer, your credit card rewards point of Rs 9,850 will expire today. Redeem now into your bank account: http://redeem-card.com",
        "Verification needed: Your account has suspicious login attempts. Verify identity within 24 hours to prevent lock.",
        "Verify your UPI ID to claim the cash back reward of Rs 2,000 immediately: http://cashback-upi.in",
        "ALERT: Debit card blocked. SMS KYC to activate or call support desk immediately.",
        "Get instant personal loan of up to Rs 5 Lakhs with zero documentation. Click link to download APK: http://loan-apk.net",
        "Urgent: Share the 6-digit verification code sent to your phone to secure your WhatsApp account immediately."
    ],
    'label': [
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1
    ]
}

df = pd.DataFrame(data)
df = pd.concat([df] * 12, ignore_index=True)

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Data split: {len(X_train)} training, {len(X_test)} testing messages.")


## 3. Model Pipeline & Training
We vectorize text using **TF-IDF** (with standard English stop-words filtering) and train a **Multinomial Naive Bayes** classifier, which is extremely efficient for sparse text counts.

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print("Validation Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


## 4. Exporting the Pipeline
We bundle the TF-IDF vectorizer and the Multinomial Naive Bayes weights together into a single pickle file `message_classifier.pkl` in the `python/` directory.

In [ ]:
output_dir = '../python'
os.makedirs(output_dir, exist_ok=True)
model_path = os.path.join(output_dir, 'message_classifier.pkl')
joblib.dump({'vectorizer': vectorizer, 'model': model}, model_path)
print(f"Message classifier pipeline successfully exported to {model_path}")
